# Checking the toxicity of political comments

In [1]:
!pip install detoxify pandas torch
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 87.6 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 3.6 MB/s  0:00:43m0:00:0100:11m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 99.1 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 99.6 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 96.6 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 98.1 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 97.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 90.1 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 84.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 2.0 MB/s  0:00:45m0:00:0100:020m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 12.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/

In [2]:
import pandas as pd
import numpy as np
from detoxify import Detoxify
from tqdm import tqdm
from transformers import pipeline

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("/home/onyxia/work/Reddit-Polarization/annotations/toxicity/data_1_political_final.csv")

In [5]:
df

,id,user,date_ins,type,date,text,user_inter,pfp_inter,text_inter,post_title,post_text,after,language,data_1_clean_political
0,1X000149,Yare_Yare_Daze-I,NaN,comment,2020-06-01 11:49:15+00:00,LawlThe ss is bad communauty don't agree with U,Sightshade,NaN,post body,A much-needed reminder to be civil and respect...,It’s been a few days since the Origami King re...,t1_fsinitc,en,1
1,1X000757,topiga,NaN,comment,2026-01-30 09:48:26+00:00,Fin des partiels pour ma copine. Elle a changé...,AutoModerator,NaN,post body,Forum Libre - 2026-01-30,Partagez ici tout ce que vous voulez sauf la p...,t1_ny7xgij,fr,1
2,1X0007228,topiga,NaN,comment,2025-11-05 09:45:51+00:00,EU 30,qidi_3dprinter,NaN,post body,🎃 QIDI Halloween Treasure Hunt Giveaway! 👻,It’s time for some fun! Many pumpkins are wait...,t1_nlsoxsz,en,1
3,1X0007231,topiga,NaN,comment,2025-11-05 08:38:57+00:00,EU 30,qidi_3dprinter,NaN,post body,🎃 QIDI Halloween Treasure Hunt Giveaway! 👻,It’s time for some fun! Many pumpkins are wait...,t1_nlsoxsz,en,1
4,1X0007255,topiga,NaN,comment,2025-10-23 05:43:33+00:00,I don’t agree with everything. It’s nice to kn...,[deleted],NaN,post body,[deleted by user],[removed],t3_1nihtpj,en,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37102,1X18298,Osarel,NaN,comment,2025-09-08 15:32:18+00:00,Euh ça va ? C'est le 3ème sujet sur la remise ...,99ShahedOfBakuOfNine,NaN,post body,Le droit de vote devrait sauter pour ceux qui ...,Tout est dans le titre : le groupe minoritaire...,End,fr,1
37103,1X182912,Osarel,NaN,comment,2025-09-01 13:04:13+00:00,Pour avoir eu un père alcoolique et une mère t...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1
37104,1X182914,Osarel,NaN,comment,2025-07-18 07:26:55+00:00,Bof. Tu retiens surtout la douleur et ensuite ...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1
37105,1X182915,Osarel,NaN,comment,2025-06-21 13:50:41+00:00,C'est effectivement impopulaire je pense. Moi ...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1


## Detoxify (trop long, plutôt partie suivante)

In [ ]:
model = Detoxify(
    'multilingual',
    device='cpu'
)

batch_size = 64

toxicity_scores = []

comments = df["text"].fillna("").tolist()

for i in tqdm(range(0, len(comments), batch_size)):
    batch = comments[i:i+batch_size]

    results = model.predict(batch)

    toxicity_scores.extend(results["toxicity"])

df["toxicity"] = toxicity_scores

Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.4-alpha/multilingual_debiased-0b549669.ckpt" to /home/onyxia/.cache/torch/hub/checkpoints/multilingual_debiased-0b549669.ckpt


100%|██████████| 1.04G/1.04G [00:10<00:00, 102MB/s] 
100%|██████████| 580/580 [1:42:06<00:00, 10.56s/it]


In [6]:
df.to_csv("data_1_toxicity.csv", index=False, encoding="utf-8")

In [19]:
df

,id,user,date_ins,type,date,text,user_inter,pfp_inter,text_inter,post_title,post_text,after,language,data_1_clean_political,toxicity
0,1X000149,Yare_Yare_Daze-I,NaN,comment,2020-06-01 11:49:15+00:00,LawlThe ss is bad communauty don't agree with U,Sightshade,NaN,post body,A much-needed reminder to be civil and respect...,It’s been a few days since the Origami King re...,t1_fsinitc,en,1,0.753615
1,1X000757,topiga,NaN,comment,2026-01-30 09:48:26+00:00,Fin des partiels pour ma copine. Elle a changé...,AutoModerator,NaN,post body,Forum Libre - 2026-01-30,Partagez ici tout ce que vous voulez sauf la p...,t1_ny7xgij,fr,1,0.001628
2,1X0007228,topiga,NaN,comment,2025-11-05 09:45:51+00:00,EU 30,qidi_3dprinter,NaN,post body,🎃 QIDI Halloween Treasure Hunt Giveaway! 👻,It’s time for some fun! Many pumpkins are wait...,t1_nlsoxsz,en,1,0.000452
3,1X0007231,topiga,NaN,comment,2025-11-05 08:38:57+00:00,EU 30,qidi_3dprinter,NaN,post body,🎃 QIDI Halloween Treasure Hunt Giveaway! 👻,It’s time for some fun! Many pumpkins are wait...,t1_nlsoxsz,en,1,0.000452
4,1X0007255,topiga,NaN,comment,2025-10-23 05:43:33+00:00,I don’t agree with everything. It’s nice to kn...,[deleted],NaN,post body,[deleted by user],[removed],t3_1nihtpj,en,1,0.010784
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37102,1X18298,Osarel,NaN,comment,2025-09-08 15:32:18+00:00,Euh ça va ? C'est le 3ème sujet sur la remise ...,99ShahedOfBakuOfNine,NaN,post body,Le droit de vote devrait sauter pour ceux qui ...,Tout est dans le titre : le groupe minoritaire...,End,fr,1,0.000552
37103,1X182912,Osarel,NaN,comment,2025-09-01 13:04:13+00:00,Pour avoir eu un père alcoolique et une mère t...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1,0.188919
37104,1X182914,Osarel,NaN,comment,2025-07-18 07:26:55+00:00,Bof. Tu retiens surtout la douleur et ensuite ...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1,0.301428
37105,1X182915,Osarel,NaN,comment,2025-06-21 13:50:41+00:00,C'est effectivement impopulaire je pense. Moi ...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1,0.007133


## Hugging Face CitizenLab

df_2 peut être remplacé par le dataset complet avec les bords politiques, ou bien joint a posterori dessus.

In [8]:
df_2 = pd.read_csv("/home/onyxia/work/Reddit-Polarization/annotations/toxicity/data_2_political_final.csv")

In [9]:
classifier = pipeline(
    "text-classification",
    model="citizenlab/distilbert-base-multilingual-cased-toxicity",
    batch_size=64,
    device=-1
)

comments = df_2["text"].fillna("").tolist()

results = []

for i in tqdm(range(0, len(comments), 64)):
    batch = comments[i:i+64]

    preds = classifier(
        batch,
        truncation=True,
        max_length=256
    )

    results.extend(preds)

df_2["toxicity_score"] = [x["score"] for x in results]
df_2["toxicity_label"] = [x["label"] for x in results]

100%|██████████| 562/562 [31:49<00:00,  3.40s/it]


In [11]:
df_2.to_csv("data_2_toxicity.csv", index=False, encoding="utf-8")

In [3]:
df_1 = pd.read_csv("/home/onyxia/work/Reddit-Polarization/annotations/toxicity/data_1_political_final.csv")

In [13]:
classifier = pipeline(
    "text-classification",
    model="citizenlab/distilbert-base-multilingual-cased-toxicity",
    batch_size=64,
    device=-1
)

comments = df_1["text"].fillna("").tolist()

results = []

for i in tqdm(range(0, len(comments), 64)):
    batch = comments[i:i+64]

    preds = classifier(
        batch,
        truncation=True,
        max_length=256
    )

    results.extend(preds)

df_1["toxicity_score"] = [x["score"] for x in results]
df_1["toxicity_label"] = [x["label"] for x in results]


df_1.to_csv("data_1_toxicity_hf.csv", index=False, encoding="utf-8")

100%|██████████| 580/580 [33:49<00:00,  3.50s/it]


In [18]:
df_2["toxicity_label"].value_counts()

toxicity_label
not_toxic    33952
toxic         1978
Name: count, dtype: int64

In [4]:
df_1 = pd.read_csv("data_1_toxicity_hf.csv")

In [5]:
# A faire pour tous les datasets, on converti le score par rapport aux labels, en score de toxicité
df_1["toxicity_level"] = np.where(
    df_1["toxicity_label"] == "toxic",
    df_1["toxicity_score"],
    1 - df_1["toxicity_score"]
)

In [6]:
df_1.to_csv("data_1_toxicity_hf.csv", index=False, encoding="utf-8")

In [7]:
df_2 = pd.read_csv("data_2_toxicity_hf.csv")

In [8]:
df_2["toxicity_level"] = np.where(
    df_2["toxicity_label"] == "toxic",
    df_2["toxicity_score"],
    1 - df_2["toxicity_score"]
)

In [9]:
df_2.to_csv("data_2_toxicity_hf.csv", index=False, encoding="utf-8")

In [10]:
df_2

,id,user,date_ins,type,date,text,user_inter,pfp_inter,text_inter,post_title,post_text,after,language,data_2_clean_political,toxicity_score,toxicity_label,toxicity_level
0,2X00023,Spirited-Researcher1,NaN,comment,2025-12-27 21:48:31+00:00,Le problème est simplement que personne ne sai...,Grin-Guy,NaN,post body,"Logement : les biens sans maître, ce « fléau d...",NaN,t1_ldb4vj7,fr,1,0.995470,not_toxic,0.004530
1,2X00028,Spirited-Researcher1,NaN,comment,2025-09-28 04:44:54+00:00,"Also check your contract, sometimes if it is s...",FrenchyQV,NaN,post body,Solution to make extra money in the week ends?,Hello Reddit! \n\nI have a full time contract ...,t1_ldb4vj7,en,1,0.999103,not_toxic,0.000897
2,2X000253,Spirited-Researcher1,NaN,response,2024-07-11 04:41:59+00:00,Je les soupçonne d’avoir enlevé la femelle fél...,Overall-Link-7546,NaN,Ouais mais eux ils sont en finale 😣,Are you proud to be British?,NaN,t1_isdy5y4,fr,1,0.969612,not_toxic,0.030388
3,2X000271,Spirited-Researcher1,NaN,response,2024-01-28 05:45:53+00:00,"After I lost a first empire , I started to plo...",realshockvaluecola,NaN,One time I lost TWO empires to the Mongols. I ...,Never Knew!!! Mongols can invade empires,NaN,t1_isdy5y4,en,1,0.998084,not_toxic,0.001916
4,2X0002107,Spirited-Researcher1,NaN,response,2022-06-11 21:17:58+00:00,… le train de tes insultes roule sur les rails...,Zeroleouf,NaN,"Ecoute moi bien mon petit Jose, tu baises les ...",Un dénomé José,NaN,t1_h2fuq7i,fr,1,0.976593,not_toxic,0.023407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35925,2X188339,Worldly_Safety,NaN,response,2025-06-18 13:16:59+00:00,Certains parlent de perte de savoir faire en o...,Down_Badger_2253,NaN,*The text of this post is no longer accessible...,Avenir de l'action LVMH,"Hello,Il y a-t-il des investisseurs LVMH —acti...",t1_my2tlsv,fr,1,0.995559,not_toxic,0.004441
35926,2X188343,Worldly_Safety,NaN,response,2025-06-17 15:54:23+00:00,Le bêta slippage fait parti des risques à pren...,Nuppys,NaN,Je parlais de l'effet de levier sur un portefe...,Les français et la bourse en 2025,https://www.lemonde.fr/argent/article/2025/06/...,t1_my2tlsv,fr,1,0.996535,not_toxic,0.003465
35927,2X188344,Worldly_Safety,NaN,comment,2025-06-17 13:17:11+00:00,"""Jeune"" de 30 ans ici, investi principalement ...",Nuppys,NaN,post body,Les français et la bourse en 2025,https://www.lemonde.fr/argent/article/2025/06/...,t1_my2tlsv,fr,1,0.996454,not_toxic,0.003546
35928,2X188345,Worldly_Safety,NaN,response,2025-06-16 21:34:23+00:00,J'suis désolé mais ce raisonnement c'est juste...,Moist_Pack_6399,NaN,J'ai toujours du mal à comprendre comment c'es...,"Pour sauver ses voitures, ses médicaments et a...",NaN,t1_my2tlsv,fr,1,0.991811,not_toxic,0.008189


In [13]:
df_toxic = pd.concat([df_1, df_2])
df_toxic

,id,user,date_ins,type,date,text,user_inter,pfp_inter,text_inter,post_title,post_text,after,language,data_1_clean_political,toxicity_score,toxicity_label,toxicity_level,data_2_clean_political
0,1X000149,Yare_Yare_Daze-I,NaN,comment,2020-06-01 11:49:15+00:00,LawlThe ss is bad communauty don't agree with U,Sightshade,NaN,post body,A much-needed reminder to be civil and respect...,It’s been a few days since the Origami King re...,t1_fsinitc,en,1.0,0.995048,not_toxic,0.004952,NaN
1,1X000757,topiga,NaN,comment,2026-01-30 09:48:26+00:00,Fin des partiels pour ma copine. Elle a changé...,AutoModerator,NaN,post body,Forum Libre - 2026-01-30,Partagez ici tout ce que vous voulez sauf la p...,t1_ny7xgij,fr,1.0,0.512768,not_toxic,0.487232,NaN
2,1X0007228,topiga,NaN,comment,2025-11-05 09:45:51+00:00,EU 30,qidi_3dprinter,NaN,post body,🎃 QIDI Halloween Treasure Hunt Giveaway! 👻,It’s time for some fun! Many pumpkins are wait...,t1_nlsoxsz,en,1.0,0.997579,not_toxic,0.002421,NaN
3,1X0007231,topiga,NaN,comment,2025-11-05 08:38:57+00:00,EU 30,qidi_3dprinter,NaN,post body,🎃 QIDI Halloween Treasure Hunt Giveaway! 👻,It’s time for some fun! Many pumpkins are wait...,t1_nlsoxsz,en,1.0,0.997579,not_toxic,0.002421,NaN
4,1X0007255,topiga,NaN,comment,2025-10-23 05:43:33+00:00,I don’t agree with everything. It’s nice to kn...,[deleted],NaN,post body,[deleted by user],[removed],t3_1nihtpj,en,1.0,0.998865,not_toxic,0.001135,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35925,2X188339,Worldly_Safety,NaN,response,2025-06-18 13:16:59+00:00,Certains parlent de perte de savoir faire en o...,Down_Badger_2253,NaN,*The text of this post is no longer accessible...,Avenir de l'action LVMH,"Hello,Il y a-t-il des investisseurs LVMH —acti...",t1_my2tlsv,fr,NaN,0.995559,not_toxic,0.004441,1.0
35926,2X188343,Worldly_Safety,NaN,response,2025-06-17 15:54:23+00:00,Le bêta slippage fait parti des risques à pren...,Nuppys,NaN,Je parlais de l'effet de levier sur un portefe...,Les français et la bourse en 2025,https://www.lemonde.fr/argent/article/2025/06/...,t1_my2tlsv,fr,NaN,0.996535,not_toxic,0.003465,1.0
35927,2X188344,Worldly_Safety,NaN,comment,2025-06-17 13:17:11+00:00,"""Jeune"" de 30 ans ici, investi principalement ...",Nuppys,NaN,post body,Les français et la bourse en 2025,https://www.lemonde.fr/argent/article/2025/06/...,t1_my2tlsv,fr,NaN,0.996454,not_toxic,0.003546,1.0
35928,2X188345,Worldly_Safety,NaN,response,2025-06-16 21:34:23+00:00,J'suis désolé mais ce raisonnement c'est juste...,Moist_Pack_6399,NaN,J'ai toujours du mal à comprendre comment c'es...,"Pour sauver ses voitures, ses médicaments et a...",NaN,t1_my2tlsv,fr,NaN,0.991811,not_toxic,0.008189,1.0


In [11]:
df_full = pd.read_csv("flat_political_interactions.csv")

In [12]:
df_full

,text
0,my parents are kurdish immigrants that moved t...
1,On voit partout sur les réseaux des publicatio...
2,"Les coiffeurs ne figurant pas, à ma connaissan..."
3,L'Établissement Français du Sang n'a pas inter...
4,"On peut accuser internet, les films, les jeux ..."
...,...
151132,Tu confonds public et privé.Un fois de plus ce...
151133,Je ne pense pas que l'idée soit de le défendre...
151134,"Non, ils sont vraiment comme ça.J'en ai vu qui..."
151135,Il avait son fan club dès le départ juste parc...


In [14]:
df_toxic_final = df_full.merge(
    df_toxic[["text", "toxicity_level"]],
    on="text",
    how="left"
)
df_toxic_final

,text,toxicity_level
0,my parents are kurdish immigrants that moved t...,NaN
1,On voit partout sur les réseaux des publicatio...,NaN
2,"Les coiffeurs ne figurant pas, à ma connaissan...",NaN
3,L'Établissement Français du Sang n'a pas inter...,NaN
4,"On peut accuser internet, les films, les jeux ...",NaN
...,...,...
164286,Tu confonds public et privé.Un fois de plus ce...,NaN
164287,Je ne pense pas que l'idée soit de le défendre...,NaN
164288,"Non, ils sont vraiment comme ça.J'en ai vu qui...",NaN
164289,Il avait son fan club dès le départ juste parc...,NaN


In [ ]:
# Recupération des données non classifiées
df_toxic_na = df_toxic_final[df_toxic_final["toxicity_level"].isna()].drop(columns=["toxicity_level"])
df_toxic_na

,text
0,my parents are kurdish immigrants that moved t...
1,On voit partout sur les réseaux des publicatio...
2,"Les coiffeurs ne figurant pas, à ma connaissan..."
3,L'Établissement Français du Sang n'a pas inter...
4,"On peut accuser internet, les films, les jeux ..."
...,...
164286,Tu confonds public et privé.Un fois de plus ce...
164287,Je ne pense pas que l'idée soit de le défendre...
164288,"Non, ils sont vraiment comme ça.J'en ai vu qui..."
164289,Il avait son fan club dès le départ juste parc...


In [26]:
classifier = pipeline(
    "text-classification",
    model="citizenlab/distilbert-base-multilingual-cased-toxicity",
    batch_size=64,
    device=-1
)

comments = df_toxic_na["text"].fillna("").tolist()

results = []

for i in tqdm(range(0, len(comments), 64)):
    batch = comments[i:i+64]

    preds = classifier(
        batch,
        truncation=True,
        max_length=256
    )

    results.extend(preds)

df_toxic_na["toxicity_score"] = [x["score"] for x in results]
df_toxic_na["toxicity_label"] = [x["label"] for x in results]


df_toxic_na.to_csv("data_3_toxicity_hf.csv", index=False, encoding="utf-8")

100%|██████████| 1426/1426 [1:21:44<00:00,  3.44s/it]


In [27]:
# Conversion du label en toxique
df_toxic_na["toxicity_level"] = np.where(
    df_toxic_na["toxicity_label"] == "toxic",
    df_toxic_na["toxicity_score"],
    1 - df_toxic_na["toxicity_score"]
)
df_toxic_na.to_csv("data_3_toxicity_hf.csv", index=False, encoding="utf-8")

In [40]:
df_toxic_na = pd.read_csv("data_3_toxicity_hf.csv")

In [41]:
df_toxic_na


,text,toxicity_score,toxicity_label,toxicity_level
0,my parents are kurdish immigrants that moved t...,0.983277,not_toxic,0.016723
1,On voit partout sur les réseaux des publicatio...,0.996199,not_toxic,0.003801
2,"Les coiffeurs ne figurant pas, à ma connaissan...",0.756635,not_toxic,0.243365
3,L'Établissement Français du Sang n'a pas inter...,0.992593,not_toxic,0.007407
4,"On peut accuser internet, les films, les jeux ...",0.980755,not_toxic,0.019245
...,...,...,...,...
91249,Tu confonds public et privé.Un fois de plus ce...,0.996985,not_toxic,0.003015
91250,Je ne pense pas que l'idée soit de le défendre...,0.982947,not_toxic,0.017053
91251,"Non, ils sont vraiment comme ça.J'en ai vu qui...",0.994498,not_toxic,0.005502
91252,Il avait son fan club dès le départ juste parc...,0.903258,not_toxic,0.096742


In [42]:
# On rajoute les dernières valeurs
df_toxic_final = df_toxic_final.merge(
    df_toxic_na[["text", "toxicity_level"]],
    on="text",
    how="left",
    suffixes=("", "_df3")
)


In [44]:
df_toxic_final = df_toxic_final.iloc[:, :-3]

In [45]:
df_toxic_final

,text,toxicity_level,toxicity_level_df3
0,my parents are kurdish immigrants that moved t...,NaN,0.016723
1,On voit partout sur les réseaux des publicatio...,NaN,0.003801
2,"Les coiffeurs ne figurant pas, à ma connaissan...",NaN,0.243365
3,L'Établissement Français du Sang n'a pas inter...,NaN,0.007407
4,"On peut accuser internet, les films, les jeux ...",NaN,0.019245
...,...,...,...
164286,Tu confonds public et privé.Un fois de plus ce...,NaN,0.003015
164287,Je ne pense pas que l'idée soit de le défendre...,NaN,0.017053
164288,"Non, ils sont vraiment comme ça.J'en ai vu qui...",NaN,0.005502
164289,Il avait son fan club dès le départ juste parc...,NaN,0.096742


In [46]:
# Equivalent d'un coalesce
df_toxic_final["toxicity_level"] = df_toxic_final["toxicity_level"].fillna(df_toxic_final["toxicity_level_df3"])

# Supprimer la colonne temporaire
df_toxic_final = df_toxic_final.drop(columns=["toxicity_level_df3"])

In [47]:
df_toxic_final

,text,toxicity_level
0,my parents are kurdish immigrants that moved t...,0.016723
1,On voit partout sur les réseaux des publicatio...,0.003801
2,"Les coiffeurs ne figurant pas, à ma connaissan...",0.243365
3,L'Établissement Français du Sang n'a pas inter...,0.007407
4,"On peut accuser internet, les films, les jeux ...",0.019245
...,...,...
164286,Tu confonds public et privé.Un fois de plus ce...,0.003015
164287,Je ne pense pas que l'idée soit de le défendre...,0.017053
164288,"Non, ils sont vraiment comme ça.J'en ai vu qui...",0.005502
164289,Il avait son fan club dès le départ juste parc...,0.096742


In [49]:
df_toxic_final["toxicity_level"].value_counts()

toxicity_level
0.006099    34
0.000893    17
0.000865    16
0.000969    15
0.080620    15
            ..
0.095118     1
0.063015     1
0.023320     1
0.085108     1
0.004793     1
Name: count, Length: 117674, dtype: int64

In [50]:
df_toxic_final.count()

text              164291
toxicity_level    164291
dtype: int64

In [51]:
df_toxic_final

,text,toxicity_level
0,my parents are kurdish immigrants that moved t...,0.016723
1,On voit partout sur les réseaux des publicatio...,0.003801
2,"Les coiffeurs ne figurant pas, à ma connaissan...",0.243365
3,L'Établissement Français du Sang n'a pas inter...,0.007407
4,"On peut accuser internet, les films, les jeux ...",0.019245
...,...,...
164286,Tu confonds public et privé.Un fois de plus ce...,0.003015
164287,Je ne pense pas que l'idée soit de le défendre...,0.017053
164288,"Non, ils sont vraiment comme ça.J'en ai vu qui...",0.005502
164289,Il avait son fan club dès le départ juste parc...,0.096742


In [52]:
df_toxic_final.to_csv("political_interactions_toxic.csv")